# 资源分配问题

In [288]:
import numpy as np
import copy
from time import time

## 类定义
定义两个类，node表示节点类， net表示线网类

In [289]:
class node():
    def __init__(self, ind=None, name=None, ziyuan=None, assignment=None):
        self.id = ind
        self.name = name
        self.ziyuan = ziyuan         # 节点所用资源的列表(在当前题目要求下，资源总和即可，不需要列表)
        self.assignment = assignment  # 节点被分配到的FPGA序号
        self.net = []               # 节点所属线网的序号列表


class net():
    def __init__(self, nodelist=None):
        self.nodelist = nodelist        # 线网中的节点的列表，此列表中每个元素是节点的名字
        self.assignment = set()    # 线网中的节点被分配到的FPGA的集合， 此集合中每个元素是FPGA的序号


## 定义函数
### 文件信息读取
读取文件，取得节点资源信息

In [290]:
def readNodeFile(filename):
    f = open(filename)
    vertex = {}        # 字典
    vertexname=[]      # 列表
    ind = 0
    for line in f:
        list1 = line.split()
        nm = list1[0]
        vertexname.append(nm)
        ziyuan = list(map(int, list1[-10:]))
        vertex[nm] = node(ind, nm, sum(ziyuan))
        ind += 1
    f.close()
    return vertex,vertexname

读取文件，取得线网信息

In [291]:

def readNetFile(filename,vertex):
    f = open(filename)
    linestack=[]
    for line in f:
        linestack.append(line.split())
    nets=[]
    nodelist=[]
    ind=0
    while linestack!=[]:
        line = linestack.pop()
        nodelist.append(line[0])
        if 's' in line:
            nets.append(net(nodelist))
            for node in nodelist:
                vertex[node].net.append(ind)
            nodelist=[]
            ind += 1
    f.close()
    return nets

### 定义分配函数fenpeiMethod

In [292]:
def fenpeiMethod():
    nosnode = len(vertex)
    fenpei=[]
    for i in range(nosnode):
        fenpei.append(np.random.randint(nosFPGA))
    F = [[] for i in range(nosFPGA)]
    for i in range(nosnode):
        F[fenpei[i]].append(i)
        vertex[vertexname[i]].assignment = fenpei[i]
        for onenet in pop:
            if vertexname[i] in onenet.nodelist:
                onenet.assignment.add(fenpei[i])
    return F

### 定义写文件函数writeResult

In [293]:
def writeResult(F,vertexname):
    nosFPGA = len(F)
    for fpga in range(nosFPGA):
        f.write("F"+str(fpga))
        for onenode in F[fpga]:
            f.write(" "+vertexname[onenode])
        f.write("\n")

### 定义资源差异函数 ziyuanCal
计算各个板的资源

In [294]:
def ziyuanCal(F,vertex,vertexname):
    nosFPGA = len(F)
    Z = []
    for i in range(nosFPGA):
        list1 = F[i]
        print(list1)
        sumzy = 0
        for ind in list1:
            sumzy += vertex[vertexname[ind]].ziyuan
        Z.append(sumzy)
    return Z


### 定义板间连线计算函数 linkCal
计算FPGA板间连线的总和

In [295]:
def linkCal(nets):
    L = [0 for i in range(nosFPGA)]
    for onenet in nets:
        if len(onenet.assignment)==1:
            continue
        for fpgaInd in onenet.assignment:
            L[fpgaInd] += 1
    return L


### 清空分配信息
包括节点和线网中的分配信息

In [296]:
def clearAssignInfo(nets):
    for nodeInd in range(len(vertexname)):
        vertex[vertexname[nodeInd]].assignment = None
    for onenet in nets:
        onenet.assignment = set()

### 遗传算法

初始化染色体群体

In [297]:
# 产生N个染色体的初始群体,保存在pop里
# 每个染色体代表连线问题的某一个解（每个节点和板块和分配关系）

def initPop(N):
    pop = []
    # pop.append(fenpeiMethod() for _ in range(N))
    for i in range(N):  # 使用随机函数生成N个染色体
        pop.append(fenpeiMethod())
    return pop


计算板间连线长度

In [298]:
# 计算pop中每个染色体所代表的分配关系的板间连线长度

def calLength(pop):
    length =[]
    for i in pop:
        length.append(linkCal(i))
    return length


In [299]:
# 根据染色体群体pop已经对应的适应值fitness，
# 找到最高的适应值f，以及对应的染色体bst和其在pop中的编号/下标ind

def findBest(pop, fitness):
    f = max(fitness)
    index = fitness.index(f)
    best = pop[index]
    return [best, f, index]


In [300]:
# 根据染色体的适应值，按照一定的概率，生成新一代染色体群体newpop

def chooseNewP(pop, fitness):
    N, nc = np.shape(pop)
    fitness = np.cumsum(fitness)
    lst = np.zeros([N, 1])
    rvalLst = np.random.rand(N, 1)
    for i in range(N):
        rval = np.random.rand()
        lst[i] = 0
        for j in range(N-1, -1, -1):
            if rval > fitness[j]:
                lst[i] = j
                break
    newpop = pop[list(lst.flatten().astype(np.uint8)), :]
    return newpop


## 执行代码
读取文件

In [301]:
# 读取节点文件
filename = "./contest/testdata-0/design.are"
vertex,vertexname = readNodeFile(filename)

In [302]:
# 读取线网文件
filename = "./contest/testdata-0/design.net"
pop = readNetFile(filename,vertex)

In [303]:
print(pop[3].nodelist)
print(pop[4].assignment)

['g25', 'g24', 'g23', 'g26']
set()


超参数设置：

In [304]:
nosFPGA = 4 	# FPGA的个数
N = 10		# 染色体群体规模
MAXITER = 10**3  # 最大循环次数
pc = 0.15		# 交叉概率
pw = 0.85		# 变异概率


把结果文件打开

In [305]:
filename = "./result.res"
f = open(filename,'w')

第1种情况：

In [306]:
# clearAssignInfo(vertex,vertexname,nets)
# F = fenpeiMethod(vertex,nosFPGA,vertexname)
# writeResult(F,vertexname)
# f.write(str(np.std(ziyuanCal(F,vertex,vertexname)))+"\n")

第2种情况：

In [307]:
# clearAssignInfo(nets)
pop = initPop(N)
tstart = time()
for iter in range(MAXITER):
    length = calLength(pop)  # 步骤2：计算每个染色体的连线长度
    fitness = 1.0/length  # 计算每个染色体的适应值
    best, f, index = findBest(pop, fitness)			# 找到在当前种群中，适应度最高的个体
    chooseNewP(pop, fitness)  # 否则，选取出一个新的群体
    # crossPop(pop, pc, fitness, SelfAdj)  # 步骤4：交叉产生新染色体，得到新群体
    # mutPop(pop, pw, fitness, SelfAdj)  # 步骤5：基因变异
    pop[0, :] = best[0, :]						# 保留上一代中适应值最高的染色体

    if np.mod(iter, MAXITER/10) == 1:
        titer = time()
        print("迭代次数 = %d 在运行 %d 秒后，连线长度 = %f" %
              (iter, int(titer-tstart), calLength(best[0, :])))

# 输出最优染色体/路径
print('最优的路径长度为：%f，适应值为：%f，对应的路径为：' % (calLength(best[0, :]), f))
print(best[0, :])


# 计算运行时间
tend = time()
print('The program need :  %f seconds.' % (tend-tstart))

writeResult(best, vertexname)
f.write(str(sum(linkCal(pop, nosFPGA)))+"\n")


AttributeError: 'list' object has no attribute 'assignment'

第3种情况：

In [ ]:
clearAssignInfo(vertex,vertexname,pop)
F = fenpeiMethod(vertex,nosFPGA,vertexname)
writeResult(F,vertexname)
f.write(str(sum(linkCal(pop,nosFPGA)))+"\n")

关闭文件

In [ ]:
f.close()

## 各种测试语句：

In [ ]:
print(linkCal(nets,nosFPGA))

TypeError: linkCal() takes 1 positional argument but 2 were given

In [ ]:
print(F)

[[1, 7, 10, 11, 24, 25, 38, 39, 40, 43, 44, 45, 47, 48, 49], [0, 3, 9, 16, 19, 21, 33, 37, 46, 52], [13, 17, 20, 28, 29, 31, 35, 50], [2, 4, 5, 6, 8, 12, 14, 15, 18, 22, 23, 26, 27, 30, 32, 34, 36, 41, 42, 51]]


: 

In [ ]:
print(ziyuanCal(F,vertex,vertexname))

[1, 7, 10, 11, 24, 25, 38, 39, 40, 43, 44, 45, 47, 48, 49]
[0, 3, 9, 16, 19, 21, 33, 37, 46, 52]
[13, 17, 20, 28, 29, 31, 35, 50]
[2, 4, 5, 6, 8, 12, 14, 15, 18, 22, 23, 26, 27, 30, 32, 34, 36, 41, 42, 51]
[93, 140, 30, 234]


: 

In [ ]:
np.std([276, 100, 24, 97])

92.74797841462637

: 

In [ ]:
f=[2, 7, 24, 26, 36, 43, 45, 49, 52]
[vertex[vertexname[ind]].ziyuan for ind in f]

[65, 6, 6, 3, 1, 1, 1, 1, 1]

: 

In [ ]:
sum([65, 6, 6, 3, 1, 1, 1, 1, 1])

85

: 

In [ ]:
f=[5, 6, 10, 14, 15, 16, 19, 27, 30, 31, 32, 40, 41, 44, 46]
sum([vertex[vertexname[ind]].ziyuan for ind in f])

110

: 

In [ ]:
f=[0, 1, 9, 11, 13, 17, 18, 25, 28, 29, 33, 34, 35, 37, 38, 42]
sum([vertex[vertexname[ind]].ziyuan for ind in f])

162

: 

In [ ]:
f=[3, 4, 8, 12, 20, 21, 22, 23, 39, 47, 48, 50, 51]
sum([vertex[vertexname[ind]].ziyuan for ind in f])

140

: 

In [ ]:
np.std([85,110,162,140])

29.22648627529488

: 

: 